In [ ]:
import sys
from pathlib import Path
from time import perf_counter

import joblib
import pandas as pd
import plotly.express as px
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score, f1_score

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
EVALUATION_FILE = ROOT / "results" / "model_evaluation" / "evaluation_summary.csv"
OUTPUT_DIR = ROOT / "results" / "permutation_importance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

SCORING = ["f1_macro", "balanced_accuracy"]
N_REPEATS = 5
RANDOM_STATE = 1
N_JOBS = 1


In [ ]:
evaluation_df = pd.read_csv(EVALUATION_FILE)

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_test.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features
del reference_df

columns_to_load = feature_columns + ["label_binary"]
test_dataframes = {
    dataset_name: pd.read_parquet(
        DATA_DIR / f"{dataset_name}_test.parquet",
        columns=columns_to_load,
    )
    for dataset_name in sorted(evaluation_df["test_source"].unique())
}



In [ ]:
results = []
total_start = perf_counter()

for evaluation_number, evaluation in enumerate(evaluation_df.itertuples(index=False), start=1):
    model_path = ROOT / evaluation.model_file
    if not model_path.exists():
        raise FileNotFoundError(f"modelo não encontrado: {model_path}")

    pipeline = joblib.load(model_path)
    test_df = test_dataframes[evaluation.test_source]
    X_test = test_df[feature_columns]
    y_test = test_df["label_binary"].eq("ATTACK").astype("int8")

    evaluation_start = perf_counter()
    predictions = pipeline.predict(X_test)
    baseline_scores = {
        "f1_macro": f1_score(y_test, predictions, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
    }

    importance_results = permutation_importance(
        pipeline,
        X_test,
        y_test,
        scoring=SCORING,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    elapsed_seconds = perf_counter() - evaluation_start

    for metric_name, metric_result in importance_results.items():
        for feature_index, feature_name in enumerate(feature_columns):
            row = {
                "model": evaluation.model,
                "training_source": evaluation.training_source,
                "test_source": evaluation.test_source,
                "scenario_type": evaluation.scenario_type,
                "metric": metric_name,
                "feature": feature_name,
                "baseline_score": baseline_scores[metric_name],
                "importance_mean": metric_result.importances_mean[feature_index],
                "importance_std": metric_result.importances_std[feature_index],
                "n_repeats": N_REPEATS,
                "random_state": RANDOM_STATE,
                "test_rows": len(test_df),
                "evaluation_elapsed_seconds": elapsed_seconds,
                "model_file": evaluation.model_file,
            }
            for repeat_index, importance in enumerate(
                metric_result.importances[feature_index],
                start=1,
            ):
                row[f"importance_repeat_{repeat_index}"] = importance
            results.append(row)

    print(
        f"{evaluation_number:02d}/60 - {evaluation.model} - "
        f"{evaluation.training_source} - {evaluation.test_source} - "
        f"{elapsed_seconds / 60:.2f} min"
    )

total_elapsed_seconds = perf_counter() - total_start


In [ ]:
importance_df = pd.DataFrame(results).sort_values(
    ["model", "training_source", "test_source", "metric", "importance_mean"],
    ascending=[True, True, True, True, False],
)

expected_rows = len(evaluation_df) * len(SCORING) * len(feature_columns)
if len(importance_df) != expected_rows:
    raise ValueError(f"eram esperadas {expected_rows} linhas, mas foram geradas {len(importance_df)}")

output_file = OUTPUT_DIR / "permutation_importance.csv"
importance_df.to_csv(output_file, index=False)

print(f"arquivo salvo em: {output_file}")
print(f"tempo total: {total_elapsed_seconds / 60:.2f} min")
display(importance_df)


In [2]:
from pathlib import Path

import pandas as pd
import plotly.express as px

results_file = Path(
    r"C:\tcc\network-ids-ml-generalization\results\permutation_importance\permutation_importance.csv"
)
importance_df = pd.read_csv(results_file)

models = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
training_sources = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
    "cicids2017_unsw_nb15",
    "cicids2017_iot23",
    "unsw_nb15_iot23",
]
test_sources = ["cicids2017", "unsw_nb15", "iot23"]
metrics = ["f1_macro", "balanced_accuracy"]

model = models[2]
training_source = training_sources[0]
test_source = test_sources[0]
metric = metrics[0]

selected_df = importance_df.loc[
    (importance_df["model"] == model)
    & (importance_df["training_source"] == training_source)
    & (importance_df["test_source"] == test_source)
    & (importance_df["metric"] == metric)
].sort_values("importance_mean")

if selected_df.empty:
    raise ValueError("não existem resultados para a seleção informada")

figure = px.bar(
    selected_df,
    x="importance_mean",
    y="feature",
    error_x="importance_std",
    orientation="h",
    labels={
        "importance_mean": "Redução média da métrica",
        "feature": "Feature",
    },
    title=(
        f"Importância por permutação - {model} - "
        f"{training_source} - {test_source} - {metric}"
    ),
)
figure.add_vline(x=0, line_width=1, line_color="gray")
figure.update_layout(height=650, showlegend=False)
figure.show()
